# splitserve on 2 x T4: one prefill GPU + one decode GPU vs one or two monolithic servers

Qwen2.5-1.5B-Instruct on vLLM 0.31. The same mixed open-loop load (an interactive stream of 96-token prompts with 128-token answers at 6 req/s, and a long-prompt stream of 1,500-token prompts with 48-token answers at 0.5 req/s, unique prompts so prefix caching cannot hide the prefill) goes through the `splitserve` proxy in four conditions: one monolithic server, two monolithic replicas round-robin, a prefill instance plus a decode instance with KV transferred by NixlConnector (UCX, same host), and the same pair with KV transferred through a shared directory (ExampleConnector). ~10 min install, ~6 min per condition.

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total,compute_cap,driver_version --format=csv

In [ ]:
# vLLM on the Kaggle image: the cu129 release wheel with the matching PyTorch index (the default wheel targets
# CUDA 13), and the image's preinstalled torchaudio/torchcodec removed because they are built for other CUDA
# versions and vLLM imports them (torchcodec is re-pulled by vLLM's requirements, hence the second uninstall).
%pip uninstall -q -y torchaudio torchcodec
%pip install -q "https://github.com/vllm-project/vllm/releases/download/v0.31.0/vllm-0.31.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl" --extra-index-url https://download.pytorch.org/whl/cu129
%pip uninstall -q -y torchcodec
%pip install -q "git+https://github.com/NomanJafar/servellm" "tokenbench[plots] @ git+https://github.com/NomanJafar/tokenbench"
# this repo is private: its wheel is embedded in the first cell by the portfolio's kaggle_run.sh
!pip install -q /kaggle/working/wheels/*.whl
# NIXL is the production KV-transfer path; the version is the one vLLM 0.31.0 pins. If it does not import, the
# NixlConnector condition is skipped and the ExampleConnector condition still runs.
%pip install -q "nixl==1.4.1"
import os; os.makedirs("/kaggle/working/results", exist_ok=True); os.chdir("/kaggle/working")
import torch, vllm, splitserve.proxy, servellm.launch, tokenbench
assert torch.cuda.is_available() and torch.cuda.device_count() == 2, torch.cuda.device_count()
import vllm.multimodal.media.audio
try:
    import nixl; NIXL_OK = True
except Exception as e:
    NIXL_OK = False; print("nixl import failed:", repr(e)[:300])
print("vllm", vllm.__version__, "torch", torch.__version__, "GPUs", torch.cuda.device_count(), "nixl", NIXL_OK)

## Instances, proxy, load

Each condition starts its vLLM instances cold (one per GPU via `CUDA_VISIBLE_DEVICES`), starts the proxy in the matching mode, warms the engines with a short run, clears the proxy's records, then runs the two tokenbench streams in parallel for 180 s. The proxy's per-request records (prefill round-trip and handoff time) are saved alongside the tokenbench outputs.

In [ ]:
import json, subprocess, time, httpx
from servellm.launch import ServerConfig, VllmServer
from splitserve.analysis import summarise_load, summarise_records, comparison_table

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
PROXY = "http://127.0.0.1:8080"
DURATION = 180
STREAMS = {"chat": ["--levels", "6", "--prompt-tokens", "96", "--max-tokens", "128"],
           "long": ["--levels", "0.5", "--prompt-tokens", "1500", "--max-tokens", "48"]}
NIXL_P = json.dumps({"kv_connector": "NixlConnector", "kv_role": "kv_producer", "kv_load_failure_policy": "fail"})
NIXL_D = json.dumps({"kv_connector": "NixlConnector", "kv_role": "kv_consumer", "kv_load_failure_policy": "fail"})
# /tmp, not /kaggle/working: the KV files are kernel output otherwise, and run 1 produced thousands of them
SHARED = json.dumps({"kv_connector": "ExampleConnector", "kv_role": "kv_both", "kv_connector_extra_config": {"shared_storage_path": "/tmp/kvstore"}})

def instance(gpu, port, tag, extra=(), env=None):
    e = {"CUDA_VISIBLE_DEVICES": str(gpu), "UCX_NET_DEVICES": "all", "UCX_TLS": "all", **(env or {})}
    return VllmServer(ServerConfig(model=MODEL, dtype="float16", max_model_len=4096, gpu_memory_utilization=0.9, port=port, extra_args=tuple(extra)),
                      log_path=f"results/startup_{tag}_gpu{gpu}.log", env=e)

# condition -> (list of instances, proxy args)
CONDITIONS = {
    "monolithic 1xT4": lambda: ([instance(0, 8001, "mono1")], ["--mode", "replicas", "--decode", "http://127.0.0.1:8001"]),
    "monolithic x2 (round-robin)": lambda: ([instance(0, 8001, "mono2"), instance(1, 8002, "mono2")],
                                            ["--mode", "replicas", "--decode", "http://127.0.0.1:8001", "--decode", "http://127.0.0.1:8002"]),
    "1P + 1D, NixlConnector": lambda: ([instance(0, 8100, "nixl", ["--kv-transfer-config", NIXL_P], {"VLLM_NIXL_SIDE_CHANNEL_PORT": "5600"}),
                                        instance(1, 8200, "nixl", ["--kv-transfer-config", NIXL_D], {"VLLM_NIXL_SIDE_CHANNEL_PORT": "5601"})],
                                       ["--mode", "nixl", "--prefill", "http://127.0.0.1:8100", "--decode", "http://127.0.0.1:8200"]),
    "1P + 1D, ExampleConnector (shared dir)": lambda: ([instance(0, 8100, "shared", ["--kv-transfer-config", SHARED]),
                                                        instance(1, 8200, "shared", ["--kv-transfer-config", SHARED])],
                                                       ["--mode", "shared", "--prefill", "http://127.0.0.1:8100", "--decode", "http://127.0.0.1:8200"]),
}

def start_all(servers):
    # launch concurrently, then wait for each; a failure stops the rest
    import threading
    infos, errs = {}, {}
    def go(i, s):
        try: infos[i] = s.start(ready_timeout_s=1500)
        except Exception as e: errs[i] = repr(e)
    ths = [threading.Thread(target=go, args=(i, s)) for i, s in enumerate(servers)]
    for t in ths: t.start()
    for t in ths: t.join()
    if errs: raise RuntimeError(f"instance start failed: {errs}")
    return infos

def start_proxy(args, tag):
    p = subprocess.Popen(["splitserve-proxy", "--port", "8080", *args], stdout=open(f"results/proxy_{tag}.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(100):
        try:
            if httpx.get(f"{PROXY}/health", timeout=1).status_code == 200: return p
        except httpx.HTTPError: time.sleep(0.2)
    raise RuntimeError("proxy did not start")

def tokenbench(name, tag, duration, extra):
    out = f"results/load_{tag}_{name}.json"
    cmd = ["tokenbench", "sweep", "--base-url", PROXY, "--model", MODEL, "--mode", "open", "--duration", str(duration), "--warmup", "0",
           "--prompts", "unique", "--request-timeout", "120", "--out", out, *extra]
    return subprocess.Popen(cmd, stdout=open(f"results/load_{tag}_{name}.log", "w"), stderr=subprocess.STDOUT), out

def run_load(tag, duration):
    procs = {n: tokenbench(n, tag, duration, a) for n, a in STREAMS.items()}
    for n, (p, _) in procs.items():
        try: p.wait(timeout=duration + 300)
        except subprocess.TimeoutExpired: p.kill()
    return {n: out for n, (_, out) in procs.items()}

In [ ]:
runs = []
for name, make in CONDITIONS.items():
    tag = name.split(",")[0].split(" (")[0].replace(" ", "_").replace("+", "p")
    if "Nixl" in name and not NIXL_OK:
        runs.append({"condition": name, "error": "nixl not importable"}); print("SKIP", name); continue
    rec = {"condition": name, "tag": tag}
    servers, proxy = [], None
    try:
        servers, proxy_args = make()
        t0 = time.time(); infos = start_all(servers); rec["startup_s"] = time.time() - t0
        rec["startup"] = {i: vars(inf) for i, inf in infos.items()}
        proxy = start_proxy(proxy_args, tag)
        # warm the engines (Triton JIT, first transfers), then forget those requests
        subprocess.run(["tokenbench", "sweep", "--base-url", PROXY, "--model", MODEL, "--mode", "open", "--duration", "20", "--warmup", "0",
                        "--prompts", "unique", "--levels", "2", "--prompt-tokens", "600", "--max-tokens", "32", "--request-timeout", "120",
                        "--out", f"results/_warm_{tag}.json"], check=False, capture_output=True)
        httpx.get(f"{PROXY}/records", params={"clear": "1"})
        outs = run_load(tag, DURATION)
        records = httpx.get(f"{PROXY}/records", timeout=60).json()
        json.dump(records, open(f"results/records_{tag}.json", "w"))
        rec["proxy"] = summarise_records(records)
        rec["proxy_metrics"] = httpx.get(f"{PROXY}/metrics").text
        for n, out in outs.items():
            rec[n] = summarise_load(json.load(open(out)))
        print(name, "| chat ITL p50/p99/max ms:", rec["chat"]["itl_p50_ms"], rec["chat"]["itl_p99_ms"], rec["chat"]["itl_max_ms"],
              "| long TTFT p50:", rec["long"]["ttft_p50_ms"], "| out tok/s:", rec["chat"]["out_tok_s"] + rec["long"]["out_tok_s"],
              "| proxy:", {k: (round(v, 1) if isinstance(v, float) else v) for k, v in rec["proxy"].items()})
    except Exception as e:
        rec["error"] = repr(e)[:1500]; print("FAILED", name, rec["error"][:400])
    finally:
        if proxy: proxy.terminate()
        for s in servers: s.stop()
        time.sleep(5)
    runs.append(rec)
json.dump(runs, open("results/runs.json", "w"), indent=1, default=str)

## Results

In [ ]:
ok = [r for r in runs if "error" not in r]
rows = []
for r in ok:
    rows.append({"condition": r["condition"], "requests": r["chat"]["requests"] + r["long"]["requests"], "errors": r["chat"]["errors"] + r["long"]["errors"],
                 "out_tok_s": r["chat"]["out_tok_s"] + r["long"]["out_tok_s"],
                 "chat_ttft_p50_ms": r["chat"]["ttft_p50_ms"], "chat_itl_p50_ms": r["chat"]["itl_p50_ms"], "chat_itl_p99_ms": r["chat"]["itl_p99_ms"],
                 "chat_itl_max_ms": r["chat"]["itl_max_ms"], "long_ttft_p50_ms": r["long"]["ttft_p50_ms"], "long_ttft_p95_ms": r["long"]["ttft_p95_ms"],
                 "prefill_p50_ms": r["proxy"]["prefill_p50_ms"], "handoff_p50_ms": r["proxy"]["handoff_p50_ms"], "startup_s": round(r["startup_s"])})
assert rows, "every condition failed; see runs.json"
md_ = comparison_table(rows, list(rows[0].keys()))
open("results/comparison.md", "w").write(md_); json.dump(rows, open("results/comparison.json", "w"), indent=1)
from IPython.display import Markdown, display; display(Markdown(md_))
for r in runs:
    if "error" in r: print("FAILED:", r["condition"], r["error"][:300])

In [ ]:
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
labels = [r["condition"].replace(", ", "\n") for r in rows]
axes[0].bar(labels, [r["chat_itl_max_ms"] for r in rows], color="C3", label="max"); axes[0].bar(labels, [r["chat_itl_p99_ms"] for r in rows], color="C1", label="p99")
axes[0].bar(labels, [r["chat_itl_p50_ms"] for r in rows], color="C0", label="p50"); axes[0].set_title("interactive ITL (ms)"); axes[0].legend()
axes[1].bar(labels, [r["long_ttft_p50_ms"] for r in rows], color="C2"); axes[1].set_title("long-prompt TTFT p50 (ms)")
axes[2].bar(labels, [r["out_tok_s"] for r in rows], color="C4"); axes[2].set_title("output tokens/s")
for a in axes: a.tick_params(axis="x", labelsize=7); a.grid(alpha=.3, axis="y")
fig.suptitle("2 x T4, Qwen2.5-1.5B: monolithic vs disaggregated under the same mixed load")
fig.tight_layout(); fig.savefig("results/conditions.png", dpi=130)
from IPython.display import Image; display(Image("results/conditions.png"))

In [ ]:
!cd results && zip -qr ../results.zip . && ls -la ../results.zip